# Classification Lab (Logistic Regression and k Nearest Neighbors)

In [1]:
import pandas as pd

from sklearn import neighbors

# Load Titanic data set

In [2]:
#load the titanic dataset into a data frame
df = pd.read_csv('titanic.csv')

In [3]:
#Show data frame content and check the shape of your dataframe (number of rows and columns)
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


Notice that "Survived" is the classification label of the data set

# Explore data set

Show descriptive statistics of your data frame

In [4]:
#Show some info about the data frame (columns, missing values, data types)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [5]:
#Show descriptive statistics of your data frame
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


# Data Preparation

In [6]:
#shuffle data frame using the "sample" method
df=df.sample(frac=1)

In [7]:
#Remove Name, Ticket and Cabin from dataset - you can use the drop method
df=df.drop(['Name', 'Ticket', 'Cabin'],axis=1)
df

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
171,172,0,3,male,4.0,4,1,29.1250,Q
809,810,1,1,female,33.0,1,0,53.1000,S
43,44,1,2,female,3.0,1,2,41.5792,C
68,69,1,3,female,17.0,4,2,7.9250,S
787,788,0,3,male,8.0,4,1,29.1250,Q
...,...,...,...,...,...,...,...,...,...
438,439,0,1,male,64.0,1,4,263.0000,S
146,147,1,3,male,27.0,0,0,7.7958,S
613,614,0,3,male,NaN,0,0,7.7500,Q
723,724,0,2,male,50.0,0,0,13.0000,S


In [8]:
#Apply one hot encoding to "Sex" and "Embarked" and show results
sex_oh_encoded = pd.get_dummies(df['Sex'], prefix='Sex')
#sex_oh_encoded

embarked_oh_encoded = pd.get_dummies(df['Embarked'], prefix='Embarked')
embarked_oh_encoded

,Embarked_C,Embarked_Q,Embarked_S
171,0,1,0
809,0,0,1
43,1,0,0
68,0,0,1
787,0,1,0
...,...,...,...
438,0,0,1
146,0,0,1
613,0,1,0
723,0,0,1


In [9]:
#Join one hot encoded features "Sex" and "Embarked" to data frame and show result
df=df.join(sex_oh_encoded, lsuffix='_df', rsuffix='_sex_oh_encoded')
df=df.join(embarked_oh_encoded, lsuffix='_df', rsuffix='_embarked_oh_encoded')
df

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
171,172,0,3,male,4.0,4,1,29.1250,Q,0,1,0,1,0
809,810,1,1,female,33.0,1,0,53.1000,S,1,0,0,0,1
43,44,1,2,female,3.0,1,2,41.5792,C,1,0,1,0,0
68,69,1,3,female,17.0,4,2,7.9250,S,1,0,0,0,1
787,788,0,3,male,8.0,4,1,29.1250,Q,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438,439,0,1,male,64.0,1,4,263.0000,S,0,1,0,0,1
146,147,1,3,male,27.0,0,0,7.7958,S,0,1,0,0,1
613,614,0,3,male,NaN,0,0,7.7500,Q,0,1,0,1,0
723,724,0,2,male,50.0,0,0,13.0000,S,0,1,0,0,1


What is the problem with the result of this join?

Right, the features "Sex" and "Embarked" are now redundantly stored in the data set (nominal and one hot encoding)

In [10]:
#remove the original (nominal-encoded) features
df=df.drop(['Sex','Embarked'],axis=1)
df

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
171,172,0,3,4.0,4,1,29.1250,0,1,0,1,0
809,810,1,1,33.0,1,0,53.1000,1,0,0,0,1
43,44,1,2,3.0,1,2,41.5792,1,0,1,0,0
68,69,1,3,17.0,4,2,7.9250,1,0,0,0,1
787,788,0,3,8.0,4,1,29.1250,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
438,439,0,1,64.0,1,4,263.0000,0,1,0,0,1
146,147,1,3,27.0,0,0,7.7958,0,1,0,0,1
613,614,0,3,NaN,0,0,7.7500,0,1,0,1,0
723,724,0,2,50.0,0,0,13.0000,0,1,0,0,1


In [11]:
#Replace missing values for Age with mean value - you can use the methods "fillna" and "mean"
df['Age'] = df['Age'].fillna(df['Age'].mean())
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
count,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208,0.352413,0.647587,0.188552,0.086420,0.722783
std,257.353842,0.486592,0.836071,13.002015,1.102743,0.806057,49.693429,0.477990,0.477990,0.391372,0.281141,0.447876
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,22.000000,0.000000,0.000000,7.910400,0.000000,0.000000,0.000000,0.000000,0.000000
50%,446.000000,0.000000,3.000000,29.699118,0.000000,0.000000,14.454200,0.000000,1.000000,0.000000,0.000000,1.000000
75%,668.500000,1.000000,3.000000,35.000000,1.000000,0.000000,31.000000,1.000000,1.000000,0.000000,0.000000,1.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200,1.000000,1.000000,1.000000,1.000000,1.000000


Create (pandas) histograms for all (numeric) features execpt "PassengerID" using the dataframe method "hist"

In [ ]:
hist = df.iloc[:,1:].hist(bins=20,figsize=(10, 10))

# Model Building

Split data into training and test data (500 training examples) and features vs. label

In [ ]:
num_training_examples=500

#Split data to training and test test
df_train=df.iloc[:num_training_examples,:]
df_test=df.iloc[num_training_examples:,:]

#Split training data to features and label ('Survived')
train_features=df_train.drop('Survived',axis=1)
train_label=df_train['Survived']

#Split test data to features and label ('Survived')
test_features=df_test.drop('Survived',axis=1)
test_label=df_test['Survived']

#check the results
test_label

Initialize and fit a logistics regression model, using the training data

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=0,max_iter=10000).fit(train_features, train_label)

In [ ]:
#evaluate model with test data
lr.score(test_features,test_label)

Initialize and fit a kNN model, using the training data

In [ ]:
from sklearn import neighbors

knn = neighbors.KNeighborsClassifier(n_neighbors=3)
knn.fit(train_features, train_label)

In [ ]:
#evaluate model with test data
knn.score(test_features,test_label)

# Results visualization

In [ ]:
#Predict each test label using your logistic regression and knn model
lr_prediction=lr.predict(test_features)
knn_prediction=knn.predict(test_features)

In [ ]:
#Reconstruct test data in a data frame
df_test=pd.DataFrame(test_features)

#add actual label to data frame
df_test['actual_label']=test_label

#add predicted labels to data frame
df_test['lr_prediction']=lr_prediction
df_test['knn_prediction']=knn_prediction

df_test

In [ ]:
#plot actual label vs. lr-predicted lab for test data in scatter plot using Age and Fare on x/y axis
# Scatter Plot
scatter = df_test.plot.scatter(x='Age', y='Fare', 
                           c=df_test['actual_label'] == df_test['lr_prediction'].astype(int),colormap='viridis',sharex=False)

In [ ]:
#plot actual label vs. knn-predicted lab for test data in scatter plot using Age and Fare on x/y axis
# Scatter Plot
scatter = df_test.plot.scatter(x='Age', y='Fare', 
                           c=df_test['actual_label'] == df_test['knn_prediction'].astype(int),colormap='viridis',sharex=False)